# 03 - Model Comparison

Builds the assignment's required outputs entirely from what `02_experiments.
ipynb` already wrote to `results/` - the `ExperimentTracker` log and the
saved per-model/per-square predictions - so nothing here recomputes a
prediction or a metric; this notebook is presentation and interpretation of
evidence that already exists on disk.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "forecasting").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json

import matplotlib.pyplot as plt
import pandas as pd

from forecasting.tracking import ExperimentTracker
from forecasting.viz import apply_style, CATEGORICAL, INK_SECONDARY
from forecasting.utils import hardware_info

apply_style()
RESULTS_DIR = ROOT / "results"
FIG_DIR = ROOT / "figures"

with open(RESULTS_DIR / "top_squares.json") as f:
    TOP3 = json.load(f)["top3_square_ids"]

MODEL_NAMES = ["SARIMA", "GBM", "LSTM"]

log = ExperimentTracker(RESULTS_DIR / "experiment_log.csv").read_log()
final_log = log[log["phase"] == "final"].copy()
final_log

**Interpretation.** This is every `phase="final"` row `02_experiments.ipynb`
logged: one per (model, square) combination evaluated on the held-out Dec
16-22 test week. Everything below is derived from exactly these 9 rows plus
the predictions/actuals CSVs saved alongside them.

In [ ]:
# Three per-square metric tables (MAE / MAPE / RMSE x 3 models).
per_square_tables = {}
for square_id in TOP3:
    sub = final_log[final_log["square_id"] == square_id].set_index("model")[["mae", "mape", "rmse"]]
    sub = sub.loc[[m for m in MODEL_NAMES if m in sub.index]].round(3)
    per_square_tables[square_id] = sub
    sub.to_csv(RESULTS_DIR / f"metrics_square_{square_id}.csv")
    print(f"--- Square {square_id} ---")
    print(sub, "\n")

**Interpretation - stated result, not left as an exercise:** in every one of
the three tables above, **GBM has the lowest MAE and lowest MAPE**; its RMSE
is also lowest on squares 5161 and 5259, and second-lowest (by under 1%) on
square 5059, where SARIMA's RMSE (98.15) edges it out (99.16) despite GBM
still winning that square's MAE/MAPE. **SARIMA is second on every square.
LSTM is third - last - on every metric, on every square**, with roughly
10-15% higher MAE than GBM and 15-25% higher RMSE. The same ranking holding
on all three squares, despite their different absolute traffic levels and
different weekly shapes (`01_eda.ipynb`), is the evidence that this is a
property of the three modeling approaches under this project's data and
compute budget - not an artifact of picking one particular square.

In [ ]:
# Nine actual-vs-predicted plots (3 models x 3 squares).
for square_id in TOP3:
    actuals = pd.read_csv(RESULTS_DIR / f"actuals_{square_id}.csv", index_col=0, parse_dates=True)["actual"]
    for model_name in MODEL_NAMES:
        preds = pd.read_csv(RESULTS_DIR / f"predictions_{model_name.lower()}_{square_id}.csv", index_col=0, parse_dates=True)["prediction"]

        fig, ax = plt.subplots(figsize=(9, 3.5))
        ax.plot(actuals.index, actuals.values, color=INK_SECONDARY, label="Actual", linewidth=1.3)
        ax.plot(preds.index, preds.values, color=CATEGORICAL[0], label="Predicted", linewidth=1.1, alpha=0.85)
        ax.set_title(f"{model_name} - square {square_id} - Dec 16-22 one-step-ahead forecast")
        ax.set_xlabel("Date"); ax.set_ylabel("Internet traffic (a.u.)")
        ax.legend(loc="upper right", fontsize=8)
        fig.tight_layout()
        fig.savefig(FIG_DIR / f"forecast_{model_name.lower()}_{square_id}.png")
        plt.show()
        plt.close(fig)

**Interpretation.** All nine panels track the daily rise-and-fall shape
closely - expected, given how strong the daily periodicity found in
`01_eda.ipynb` is, and all three input representations capture it one way or
another (Fourier terms, calendar features, or a full-day window). The
visible differences between models are concentrated exactly where the metric
tables above predict them to be: GBM's predicted line sits closest to actual
through the sharper peaks (e.g. square 5161's Dec 22 peak, ~5,200), while
LSTM's line visibly lags entering and leaving each day's rise - consistent
with it having the highest MAE/RMSE of the three rather than a contradiction
of the tables.

In [ ]:
# Training / prediction timing, with hardware noted.
timing_df = pd.read_csv(RESULTS_DIR / "timing.csv")
hw = hardware_info()
print(json.dumps(hw, indent=2))
timing_df

**Interpretation.** Measured on the single machine described by the hardware
block above (4-core/4-thread AMD64, 11GB RAM, CPU only - no GPU used for the
LSTM), **as a single run per (model, square) combination**, not averaged or
repeated (stated explicitly here and in `02_experiments.ipynb`'s evaluation-
protocol section, per assignment Section 4-IV). Reading the table: GBM trains
in 2-4s and predicts in 6-9s across all three squares; SARIMA trains in
7-20s (still fast, once the Fourier-term reformulation in
`forecasting/sarima_forecaster.py` avoids the >19-CPU-minute literal-
seasonal-ARIMA blowup measured during development) but its one-step Kalman-
filter walk-forward prediction is the slowest step of any model here
(92-112s over the 1,008-step test week); LSTM trains slowest overall
(35-81s) and predicts fastest (2-3s, a single forward pass per step). Put
together with the accuracy tables: **LSTM's extra training cost buys no
accuracy advantage here - it is both the most expensive model to train and
the least accurate on every square**, which is the single clearest
quantitative reason to prefer GBM for this dataset and this compute budget,
independent of the qualitative reasoning below. With a larger time budget,
the natural next step is repeating each measurement 3-5 times to report
variance alongside these point estimates, which single-run timing cannot
distinguish from measurement noise.

In [ ]:
# Failure case: the test-week timestamp with the largest mean absolute error
# across all three models, for the top-traffic square.
top_square = TOP3[0]
actuals = pd.read_csv(RESULTS_DIR / f"actuals_{top_square}.csv", index_col=0, parse_dates=True)["actual"]
preds_by_model = {
    m: pd.read_csv(RESULTS_DIR / f"predictions_{m.lower()}_{top_square}.csv", index_col=0, parse_dates=True)["prediction"]
    for m in MODEL_NAMES
}

errs = pd.DataFrame({m: (actuals - preds_by_model[m]).abs() for m in MODEL_NAMES}).dropna()
worst_ts = errs.mean(axis=1).idxmax()
window = slice(worst_ts - pd.Timedelta(hours=6), worst_ts + pd.Timedelta(hours=6))

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(actuals.loc[window].index, actuals.loc[window].values, color=INK_SECONDARY, label="Actual", linewidth=1.4)
for i, m in enumerate(MODEL_NAMES):
    p = preds_by_model[m].loc[window]
    ax.plot(p.index, p.values, color=CATEGORICAL[i + 1], label=m, linewidth=1.1, alpha=0.85)
ax.set_title(f"Failure case - square {top_square} - worst mean-error window around {worst_ts}")
ax.set_xlabel("Date"); ax.set_ylabel("Internet traffic (a.u.)")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "failure_case.png")
plt.show()

print(f"Worst mean-absolute-error timestamp across all 3 models: {worst_ts}")
print(f"Per-model absolute error at {worst_ts}:")
print(errs.loc[worst_ts].sort_values())
errs.loc[window].describe()

**Interpretation - what the plot actually shows, not a hypothetical:** the
worst shared moment in the test week is **2013-12-17 15:30**, square 5161.
Ten minutes earlier the true series had dipped to 2,766 (from ~3,100-3,200
over the preceding hour); by 15:30 it had rebounded sharply to 3,518 - a
+751 move in a single 10-minute step, well outside this square's typical
step-to-step change. All three models, conditioned on the dip, predicted a
continued low value: SARIMA 3,032 (error 486), GBM 2,984 (error 533), **LSTM
2,880 (error 638, the largest of the three at this specific step)** - the
opposite of LSTM's overall ranking, where it has the *most* error on
average but here is not simply "worse everywhere," it is worse specifically
at sudden reversals its raw 144-point window doesn't flag as unusual. None
of the three input representations - a handful of Fourier harmonics, eight
lag/calendar features, or a raw day-length window - encode "this dip is
about to reverse sharply" as a knowable-in-advance signal; all three are
built to extrapolate the recent local trend, which is exactly wrong at this
specific step. This is a concrete, evidenced limitation of all three
approaches as configured here, not a weakness of one model relative to the
others - consistent with Santos et al. [1]'s finding (cited in
`02_experiments.ipynb`) that recurrent models (and, this project's evidence
suggests, the other two paradigms as well) degrade specifically during
atypical, underrepresented local patterns, even while aggregate week-level
metrics look reasonable.